# Support Odata requests in the catalog

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1223

The goal of this story is to manage odata requests through the catalog for PRIP, and return an Odata response, with a filter on publication date. This demo requests products using both the stac (usual) and odata (new) endpoint and compares the numbers of stac assets and products received in response.

In [ ]:
import os
import requests
# Init environment before running a demo notebook.
from resources.utils import *

init_demo()
# Reload the global vars again
from resources.utils import *

In [ ]:
# Define 2 dates to filter data on

d1 = "2026-06-01T00:00:00Z"
d2 = "2026-07-01T00:00:00Z"

### Step 1: Odata request

No API endpoint was created yet, so we send the request directly.

In [ ]:
openapi_endpoint = f"/catalog/odata/v1/Products?$filter=PublicationDate gt {d1} and PublicationDate lt {d2}"
url = f"{catalog_client.href_service}{openapi_endpoint}"
headers = {'Accept': 'application/json',
          "X-API-Key": os.getenv("RSPY_APIKEY")}

openapi_response = requests.get(url, headers=headers)
odata_items = openapi_response.json()
display(odata_items)

### Step 2: equivalent STAC request

In [ ]:
stac_filter = {
    "filter": {
        "op": "and",
        "args": [
            {"op": ">=", "args": [{"property": "published"}, "2026-06-01T00:00:00Z"]},
            {"op": "<=", "args": [{"property": "published"}, "2026-06-30T00:00:00Z"]}
        ]
    },
    "filter-lang": "cql2-json"
}

stac_items = catalog_client.search(stac_filter=stac_filter)

### Step 3: check the filter

In [ ]:
[f["PublicationDate"] for f in odata_items]

### Step 4: compare stac and odata responses

We don't compare directly the number of returned items: we compare the number of stac assets (from the stac request) against the numbers of returned products (with the odata request).

In [ ]:
len(odata_items)

In [ ]:
# On the dev cluster, some features don't have assets
n_stac_assets = sum([len(f.assets) for f in stac_items if hasattr(f, "assets")])
n_stac_assets